In [1]:
import datetime
import numpy as np
import pandas as pd

from utils import (
    fetch_aeti,
    fetch_ret,
    fetch_rainfall,
    fetch_irrigations,
    fetch_kc,
    calculate_daily_water_delta,
    calculate_daily_water,
    calculate_pam_root_zone,
    calculate_pam_depletion_root_zone,
    calculate_moisture_level,
    irrigation_trigger,
)

In [2]:
# Constants
ROOT_DEPTH = np.array(([0.1] * 14) + [0.11 + 0.01 * i for i in range(90)] + [1.0] * 265) 
THETA_10 = 0.196
FIELD_CAPACITY_PER_FOOT = THETA_10 * 0.3048
PAM_FLOOR = 0.096
THETA_1500 = 0.055

In [3]:
planting_date = datetime.date(2025, 12, 10)


In [4]:

# Fetch data

aeti = fetch_aeti(planting_date)
aeti


Using cached WAPOR-3.L1-AETI-D.2025-11-D3.tif
Using cached WAPOR-3.L1-AETI-D.2025-12-D1.tif
Using cached WAPOR-3.L1-AETI-D.2025-12-D2.tif
Using cached WAPOR-3.L1-AETI-D.2025-12-D3.tif
Using cached WAPOR-3.L1-AETI-D.2026-01-D1.tif
Using cached WAPOR-3.L1-AETI-D.2026-01-D2.tif
Using cached WAPOR-3.L1-AETI-D.2026-01-D3.tif
Fetched 7 files from 2025-11-21 to 2026-01-21


,date,A-2,A-1,A-3,A-4,B-1,B-2,B-4,B-3,C-1,C-2,D-3,C-3,C-4,D-1,D-2,D-4
0,2025-11-21,0.1000,0.1,0.100000,0.1,0.100000,0.10,0.100,0.10,0.1,0.100000,0.1,0.100000,0.10,0.1,0.100000,0.100000
1,2025-11-22,0.1000,0.1,0.100000,0.1,0.100000,0.10,0.100,0.10,0.1,0.100000,0.1,0.100000,0.10,0.1,0.100000,0.100000
2,2025-11-23,0.1000,0.1,0.100000,0.1,0.100000,0.10,0.100,0.10,0.1,0.100000,0.1,0.100000,0.10,0.1,0.100000,0.100000
3,2025-11-24,0.1000,0.1,0.100000,0.1,0.100000,0.10,0.100,0.10,0.1,0.100000,0.1,0.100000,0.10,0.1,0.100000,0.100000
4,2025-11-25,0.1000,0.1,0.100000,0.1,0.100000,0.10,0.100,0.10,0.1,0.100000,0.1,0.100000,0.10,0.1,0.100000,0.100000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026-01-27,0.4375,0.6,0.466667,0.4,0.511111,0.45,0.525,0.55,0.5,0.483333,0.5,0.433333,0.35,0.5,0.383333,0.433333
68,2026-01-28,0.4375,0.6,0.466667,0.4,0.511111,0.45,0.525,0.55,0.5,0.483333,0.5,0.433333,0.35,0.5,0.383333,0.433333
69,2026-01-29,0.4375,0.6,0.466667,0.4,0.511111,0.45,0.525,0.55,0.5,0.483333,0.5,0.433333,0.35,0.5,0.383333,0.433333
70,2026-01-30,0.4375,0.6,0.466667,0.4,0.511111,0.45,0.525,0.55,0.5,0.483333,0.5,0.433333,0.35,0.5,0.383333,0.433333


In [5]:

rainfall = fetch_rainfall(planting_date)
with pd.option_context("display.max_rows", None):
    display(rainfall)

Using cached WAPOR-3.L1-PCP-E.2025-11-26.tif
Using cached WAPOR-3.L1-PCP-E.2025-11-27.tif
Using cached WAPOR-3.L1-PCP-E.2025-11-28.tif
Using cached WAPOR-3.L1-PCP-E.2025-11-29.tif
Using cached WAPOR-3.L1-PCP-E.2025-11-30.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-01.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-02.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-03.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-04.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-05.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-06.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-07.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-08.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-09.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-10.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-11.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-12.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-13.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-14.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-15.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-16.tif
Using cached WAPOR-3.L1-PCP-E.2025-12-17.tif
Using cach

,date,A-2,A-1,A-3,A-4,B-1,B-2,B-4,B-3,C-1,C-2,D-3,C-3,C-4,D-1,D-2,D-4
0,2025-11-26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00
1,2025-11-27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00
2,2025-11-28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00
3,2025-11-29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00
4,2025-11-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00
5,2025-12-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00
6,2025-12-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00
7,2025-12-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00
8,2025-12-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00
9,2025-12-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00


In [6]:

ret = fetch_ret(planting_date)
ret

Using cached WAPOR-3.L1-RET-E.2025-11-26.tif
Using cached WAPOR-3.L1-RET-E.2025-11-27.tif
Using cached WAPOR-3.L1-RET-E.2025-11-28.tif
Using cached WAPOR-3.L1-RET-E.2025-11-29.tif
Using cached WAPOR-3.L1-RET-E.2025-11-30.tif
Using cached WAPOR-3.L1-RET-E.2025-12-01.tif
Using cached WAPOR-3.L1-RET-E.2025-12-02.tif
Using cached WAPOR-3.L1-RET-E.2025-12-03.tif
Using cached WAPOR-3.L1-RET-E.2025-12-04.tif
Using cached WAPOR-3.L1-RET-E.2025-12-05.tif
Using cached WAPOR-3.L1-RET-E.2025-12-06.tif
Using cached WAPOR-3.L1-RET-E.2025-12-07.tif
Using cached WAPOR-3.L1-RET-E.2025-12-08.tif
Using cached WAPOR-3.L1-RET-E.2025-12-09.tif
Using cached WAPOR-3.L1-RET-E.2025-12-10.tif
Using cached WAPOR-3.L1-RET-E.2025-12-11.tif
Using cached WAPOR-3.L1-RET-E.2025-12-12.tif
Using cached WAPOR-3.L1-RET-E.2025-12-13.tif
Using cached WAPOR-3.L1-RET-E.2025-12-14.tif
Using cached WAPOR-3.L1-RET-E.2025-12-15.tif
Using cached WAPOR-3.L1-RET-E.2025-12-16.tif
Using cached WAPOR-3.L1-RET-E.2025-12-17.tif
Using cach

,date,A-2,A-1,A-3,A-4,B-1,B-2,B-4,B-3,C-1,C-2,D-3,C-3,C-4,D-1,D-2,D-4
0,2025-11-26,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7
1,2025-11-27,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6
2,2025-11-28,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
3,2025-11-29,2.6,2.6,2.6,2.6,2.6,2.6,2.6,2.6,2.6,2.6,2.6,2.6,2.6,2.6,2.6,2.6
4,2025-11-30,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,2026-02-07,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2
74,2026-02-08,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2
75,2026-02-09,4.5,4.5,4.5,4.5,4.5,4.5,4.5,4.5,4.5,4.5,4.5,4.5,4.5,4.5,4.5,4.5
76,2026-02-10,4.6,4.6,4.6,4.6,4.6,4.6,4.6,4.6,4.6,4.6,4.6,4.6,4.6,4.6,4.6,4.6


In [7]:
kc = fetch_kc()
kc

,date,A-1,A-2,A-3,A-4,B-1,B-2,B-3,B-4,C-1,C-2,C-3,C-4,D-1,D-2,D-3,D-4
0,2025-11-15,0.30,0.3,0.3,0.3,0.30,0.3,0.30,0.30,0.3,0.3,0.3,0.3,0.30,0.3,0.30,0.30
1,2025-11-16,0.30,0.3,0.3,0.3,0.30,0.3,0.30,0.30,0.3,0.3,0.3,0.3,0.30,0.3,0.30,0.30
2,2025-11-17,0.30,0.3,0.3,0.3,0.30,0.3,0.30,0.30,0.3,0.3,0.3,0.3,0.30,0.3,0.30,0.30
3,2025-11-18,0.30,0.3,0.3,0.3,0.30,0.3,0.30,0.30,0.3,0.3,0.3,0.3,0.30,0.3,0.30,0.30
4,2025-11-19,0.30,0.3,0.3,0.3,0.30,0.3,0.30,0.30,0.3,0.3,0.3,0.3,0.30,0.3,0.30,0.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,2026-02-24,0.63,0.3,0.3,0.3,0.58,0.3,0.61,0.64,0.3,0.3,0.3,0.3,0.63,0.3,0.60,0.60
102,2026-02-25,0.64,0.3,0.3,0.3,0.59,0.3,0.62,0.65,0.3,0.3,0.3,0.3,0.64,0.3,0.61,0.61
103,2026-02-26,0.65,0.3,0.3,0.3,0.60,0.3,0.63,0.66,0.3,0.3,0.3,0.3,0.65,0.3,0.62,0.62
104,2026-02-27,0.66,0.3,0.3,0.3,0.61,0.3,0.64,0.67,0.3,0.3,0.3,0.3,0.66,0.3,0.63,0.63


In [8]:
field_ids = list(range(37,53))
print(field_ids)
field_names = ["D-1", "B-3", "A-3", "A-2", "A-1", "D-3", "D-2", "C-2", "C-1", "C-3", "C-4", "A-4", "B-1", "B-2", "B-4", "D-4"]


[37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]


In [9]:
from dotenv import load_dotenv

load_dotenv()

irrigations = fetch_irrigations(
    field_ids=field_ids,
    field_names=field_names,
)

irrigations

,field_name,date,amount_mm
0,A-3,2025-12-25,8.0
1,B-3,2025-12-25,8.0
2,A-2,2025-12-25,8.0
3,B-1,2025-12-25,8.0
4,A-4,2025-12-23,10.0
...,...,...,...
347,B-4,2026-02-11,8.0
348,B-2,2026-02-11,8.0
349,A-2,2026-02-11,8.0
350,A-3,2026-02-11,8.0


In [10]:
# Calculate daily water delta
df = calculate_daily_water_delta(irrigations, kc, rainfall, ret)

with pd.option_context("display.max_rows", None):
    display(df)


mean ret: {'A-2': np.float64(3.6571428571428575), 'A-1': np.float64(3.6571428571428575), 'A-3': np.float64(3.6571428571428575), 'A-4': np.float64(3.6571428571428575), 'B-1': np.float64(3.6571428571428575), 'B-2': np.float64(3.6571428571428575), 'B-4': np.float64(3.6571428571428575), 'B-3': np.float64(3.6571428571428575), 'C-1': np.float64(3.6571428571428575), 'C-2': np.float64(3.6571428571428575), 'D-3': np.float64(3.6571428571428575), 'C-3': np.float64(3.6571428571428575), 'C-4': np.float64(3.6571428571428575), 'D-1': np.float64(3.6571428571428575), 'D-2': np.float64(3.6571428571428575), 'D-4': np.float64(3.6571428571428575)}
water_delta:           date   A-2   A-1   A-3   A-4   B-1   B-2   B-4   B-3   C-1   C-2  \
0   2025-11-26  1.11  1.11  1.11  1.11  1.11  1.11  1.11  1.11  1.11  1.11   
1   2025-11-27  1.08  1.08  1.08  1.08  1.08  1.08  1.08  1.08  1.08  1.08   
2   2025-11-28  0.90  0.90  0.90  0.90  0.90  0.90  0.90  0.90  0.90  0.90   
3   2025-11-29  0.78  0.78  0.78  0.78  

/Users/omar/src/autoanton/utils.py:599: UserWarning: Dates in Kc but not in RET: [datetime.date(2025, 11, 15), datetime.date(2025, 11, 16), datetime.date(2025, 11, 17), datetime.date(2025, 11, 18), datetime.date(2025, 11, 19), datetime.date(2025, 11, 20), datetime.date(2025, 11, 21), datetime.date(2025, 11, 22), datetime.date(2025, 11, 23), datetime.date(2025, 11, 24), datetime.date(2025, 11, 25), datetime.date(2026, 2, 12), datetime.date(2026, 2, 13), datetime.date(2026, 2, 14), datetime.date(2026, 2, 15), datetime.date(2026, 2, 16), datetime.date(2026, 2, 17), datetime.date(2026, 2, 18), datetime.date(2026, 2, 19), datetime.date(2026, 2, 20), datetime.date(2026, 2, 21), datetime.date(2026, 2, 22), datetime.date(2026, 2, 23), datetime.date(2026, 2, 24), datetime.date(2026, 2, 25), datetime.date(2026, 2, 26), datetime.date(2026, 2, 27), datetime.date(2026, 2, 28)]
  warnings.warn(f"Dates in Kc but not in RET: {sorted(missing_in_ret)}")
/Users/omar/src/autoanton/utils.py:612: UserWarnin

,date,A-2,A-1,A-3,A-4,B-1,B-2,B-4,B-3,C-1,C-2,D-3,C-3,C-4,D-1,D-2,D-4
0,2025-11-26,1.11,1.110,1.11,1.11,1.110,1.11,1.110,1.110,1.11,1.11,1.110,1.11,1.11,1.110,1.11,1.110
1,2025-11-27,1.08,1.080,1.08,1.08,1.080,1.08,1.080,1.080,1.08,1.08,1.080,1.08,1.08,1.080,1.08,1.080
2,2025-11-28,0.90,0.900,0.90,0.90,0.900,0.90,0.900,0.900,0.90,0.90,0.900,0.90,0.90,0.900,0.90,0.900
3,2025-11-29,0.78,0.780,0.78,0.78,0.780,0.78,0.780,0.780,0.78,0.78,0.780,0.78,0.78,0.780,0.78,0.780
4,2025-11-30,1.08,1.080,1.08,1.08,1.080,1.08,1.080,1.080,1.08,1.08,1.080,1.08,1.08,1.080,1.08,1.080
5,2025-12-01,0.99,0.990,0.99,0.99,0.990,0.99,0.990,0.990,0.99,0.99,0.990,0.99,0.99,0.990,0.99,0.990
6,2025-12-02,0.99,0.990,0.99,0.99,0.990,0.99,0.990,0.990,0.99,0.99,0.990,0.99,0.99,0.990,0.99,0.990
7,2025-12-03,0.93,0.930,0.93,0.93,0.930,0.93,0.930,0.930,0.93,0.93,0.930,0.93,0.93,0.930,0.93,0.930
8,2025-12-04,0.84,0.840,0.84,0.84,0.840,0.84,0.840,0.840,0.84,0.84,0.840,0.84,0.84,0.840,0.84,0.840
9,2025-12-05,0.84,0.840,0.84,0.84,0.840,0.84,0.840,0.840,0.84,0.84,0.840,0.84,0.84,0.840,0.84,0.840


In [11]:
df.columns

Index(['date', 'A-2', 'A-1', 'A-3', 'A-4', 'B-1', 'B-2', 'B-4', 'B-3', 'C-1',
       'C-2', 'D-3', 'C-3', 'C-4', 'D-1', 'D-2', 'D-4'],
      dtype='str')

In [ ]:
# Calculate daily water
daily_water = calculate_daily_water(daily_water_delta)

In [ ]:
# Calculate PAM root zone
pam_root_zone = calculate_pam_root_zone(daily_water, ROOT_DEPTH)

In [ ]:
# Calculate PAM depletion root zone
pam_depletion_root_zone = calculate_pam_depletion_root_zone(pam_root_zone)

In [ ]:
# Calculate moisture level and irrigation trigger
moisture_level = calculate_moisture_level(pam_root_zone)
next_irrigation = irrigation_trigger(pam_depletion_root_zone)